# *QOT Estimator Basic Usage Example*

> This notebook demonstrates the basic usage of the QOT estimator package for optical link.

In [1]:
# Import Dependencies
import sys
from pathlib import Path
import os
import numpy as np
import pandas as pd
import json
from scipy.io import loadmat

# Ignore warnings to have clean cell outputs
import warnings
warnings.filterwarnings("ignore")

### *Create results directory if it doesn't exist*

In [2]:
# Get the current working directory (where the notebook is running)
base_dir = Path.cwd()

# Define the results directory
results_dir = base_dir.parent / "results" / "single_link"

# Create it if it doesn't exist
results_dir.mkdir(exist_ok=True)

print(f"Results will be saved in: {results_dir}")

Results will be saved in: d:\Projects\CoreLight\Code\CFM_versions\CFM\results\single_link


In [3]:
sys.path.append(os.path.abspath(base_dir.parent / 'src'))
from CFM.core.network import Link, LinkParameters
from CFM.core.band import Band, OpticalParameters
from CFM.core.edfa import EdfaConfig
from CFM.core.qot_estimator import ParameterBuilder, ISRSSolver, NLISolver, ASESolver, OSNRCalculator, OSNRCalculatorV2
from CFM.core.post_process import GSNRPlotter
from CFM.utils.build_alpha_db import build_alpha_for_band

In [4]:
# Get the current working directory (where the notebook is running)
base_dir = Path.cwd()

# Define the results directory
results_dir = base_dir.parent / "results" / "single_link"

# Create it if it doesn't exist
results_dir.mkdir(exist_ok=True)

print(f"Results will be saved in: {results_dir}")

Results will be saved in: d:\Projects\CoreLight\Code\CFM_versions\CFM\results\single_link


# *QOT Estimation*

### *Create spectrum*

In [5]:
my_band_l_params =  OpticalParameters(Rs_mat = 52e9)
my_band_l = Band(
    name='l',
    start_freq = (1.845393470967740e+2 - 0.075/2), # THz
    end_freq = (1.904643470967740e+2 + 0.075/2), # THz
    opt_params = my_band_l_params,
    channel_spacing = 0.075 # THz
    )
spectrum_my_l = my_band_l.calc_spectrum() + 0.075*0.5

my_band_c_params =  OpticalParameters(Rs_mat = 52e9)
my_band_c = Band(
    name='c',
    start_freq = (1.909143470967740e+2 - 0.075/2), # THz
    end_freq = (1.968393470967740e+2 + 0.075/2), # THz
    opt_params = my_band_c_params,
    channel_spacing = 0.075 # THz
    )
spectrum_my_c = my_band_c.calc_spectrum() + 0.075*0.5

my_band_s_part1_params =  OpticalParameters(Rs_mat = 52e9)
my_band_s_part1 = Band(
    name='s',
    start_freq = (1.972893470967740e+2 - 0.075/2), # THz
    end_freq = (2.035143470967740e+2 + 0.075/2), # THz
    opt_params = my_band_s_part1_params,
    channel_spacing = 0.075 # THz
    )
spectrum_my_s_part1 = my_band_s_part1.calc_spectrum() + 0.075*0.5


my_band_s_part2_params =  OpticalParameters(Rs_mat = 52e9)
my_band_s_part2 = Band(
    name='s',
    start_freq = (2.035893870967740e+2 - 0.075/2), # THz
    end_freq = (2.053143870967740e+2 + 0.075/2), # THz
    opt_params = my_band_s_part2_params,
    channel_spacing = 0.075 # THz
    )
spectrum_my_s_part2 = my_band_s_part2.calc_spectrum() + 0.075*0.5

grid_center = np.concatenate((spectrum_my_s_part2 , spectrum_my_s_part1, spectrum_my_c , spectrum_my_l))
grid_center = grid_center[::-1]*1e12
bands = [my_band_l, my_band_c, my_band_s_part1, my_band_s_part2]

In [6]:
from CFM.utils.build_alpha_db import load_alpha_db_lcs

alpha_dB_LCS = load_alpha_db_lcs()

## *Multi Span*

### *Create fiber link*

In [7]:
my_link_params = LinkParameters()
my_link_l1 = Link(
  name='l1',
  length=[70],
  num_span=1,
  num_amp=1,
  link_params=my_link_params)

In [8]:
booster_amp = EdfaConfig(gain_target=15)
span_edfas = [
    EdfaConfig(gain_target=12.0, tilt_target=0.0),  # Span 1
    EdfaConfig(gain_target=12.0, tilt_target=0.0),                   # Span 2
    # EdfaConfig(gain_target=12.0, tilt_target=0.0, out_voa=0.0)       # Span 3
]

#### *FLP Example*

In [9]:
my_link_SNR_Param = ParameterBuilder(
  link=my_link_l1,
  bands=bands,
  P_in = [1.8],
  grid_center=grid_center,
  P_in_is_tx_power = True,
  alpha_dB_LCS=alpha_dB_LCS
)

In [10]:
my_link_ISRS = ISRSSolver(my_link_SNR_Param, 'FLP')

In [ ]:
p_in, p_out, s1, s2, sig = my_link_ISRS.solve()

In [14]:
my_link_ISRS = ISRSSolver(my_link_SNR_Param, 'FLP')

In [15]:
my_link_NLI = NLISolver(my_link_SNR_Param, s1, s2, sig)

In [16]:
nli = my_link_NLI.solve()

In [17]:
my_link_ASE = ASESolver(my_link_SNR_Param) 

In [18]:
ase = my_link_ASE.solve()

In [19]:
my_link_OSNR = OSNRCalculator(my_link_SNR_Param, ase, nli)

In [20]:
on, oa, ot = my_link_OSNR.compute()

In [21]:
ot[0]

array([33.19959323, 32.81471656, 32.64103354, 32.52857619, 32.44531526,
       32.37912145, 32.32408974, 32.27692601, 32.23560282, 32.19875239,
       32.16522236, 32.13212629, 32.10450995, 32.07971961, 32.05706722,
       32.03625408, 32.01704413, 31.9992714 , 31.98274741, 31.96735978,
       31.95296672, 31.93944812, 31.92670432, 31.91464633, 31.90309944,
       31.89167271, 31.87753694, 31.86733113, 31.85813505, 31.849653  ,
       31.84165321, 31.83414174, 31.82706927, 31.82037559, 31.81395955,
       31.80785716, 31.80191403, 31.7960925 , 31.79011331, 31.78276973,
       31.77695556, 31.77149102, 31.76607248, 31.76069419, 31.75529743,
       31.74996643, 31.74456771, 31.73917455, 31.73368061, 31.72808719,
       31.7221582 , 31.71462461, 31.70877727, 31.70328911, 31.69799524,
       31.69279048, 31.68764295, 31.68269045, 31.67778125, 31.67303054,
       31.66847705, 31.66393922, 31.6587315 , 31.65474961, 31.65124282,
       31.6481845 , 31.64561775, 31.64357998, 31.64168332, 31.64

In [22]:
curves = [
    {"name": "GSNR_total", "values": ot, "color": "blue"},
    {"name": "GSNR_linear", "values": oa, "color": "green"},
    {"name": "GSNR_nonlinear", "values": on, "color": "red"},
]

plotter = GSNRPlotter(my_link_SNR_Param, curves, bands=bands)

fig = plotter.plot(N_s_max=0, matlab_indexing=True)
fig.show()


In [23]:
my_link_OSNRV2 = OSNRCalculatorV2(my_link_SNR_Param, ase, nli)

In [24]:
res = my_link_OSNRV2.compute()

In [25]:
res['OSNR_NLI_ASE_dB']

array([[33.19959323, 32.81471656, 32.64103354, 32.52857619, 32.44531526,
        32.37912145, 32.32408974, 32.27692601, 32.23560282, 32.19875239,
        32.16522236, 32.13212629, 32.10450995, 32.07971961, 32.05706722,
        32.03625408, 32.01704413, 31.9992714 , 31.98274741, 31.96735978,
        31.95296672, 31.93944812, 31.92670432, 31.91464633, 31.90309944,
        31.89167271, 31.87753694, 31.86733113, 31.85813505, 31.849653  ,
        31.84165321, 31.83414174, 31.82706927, 31.82037559, 31.81395955,
        31.80785716, 31.80191403, 31.7960925 , 31.79011331, 31.78276973,
        31.77695556, 31.77149102, 31.76607248, 31.76069419, 31.75529743,
        31.74996643, 31.74456771, 31.73917455, 31.73368061, 31.72808719,
        31.7221582 , 31.71462461, 31.70877727, 31.70328911, 31.69799524,
        31.69279048, 31.68764295, 31.68269045, 31.67778125, 31.67303054,
        31.66847705, 31.66393922, 31.6587315 , 31.65474961, 31.65124282,
        31.6481845 , 31.64561775, 31.64357998, 31.6

### Quality of Transmission (QoT) Estimator: Multi-Band Link Tutorial

Demonstrates multi-band (**C, L, S, E**) optical link modeling using the Closed-Form Model (CFM) framework:

* **Spectrum & Fiber:** Dynamic attenuation via `build_alpha_for_band` across 4 bands.
* **Transmission:** Single-span & multi-span cascades (booster, inline EDFAs, and tilt compensation).
* **Solvers:** ISRS power evolution, analytical NLI, ASE noise accumulation, and GSNR curves.

In [27]:
import sys
from pathlib import Path
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# Setup project source imports relative to workspace root
base_dir = Path.cwd()
src_dir = base_dir.parent / "src"
if src_dir.exists() and str(src_dir) not in sys.path:
    sys.path.append(str(src_dir))

from CFM.core.network import Link, LinkParameters
from CFM.core.band import Band, OpticalParameters
from CFM.core.edfa import EdfaConfig
from CFM.core.qot_estimator import (
    ParameterBuilder, 
    ISRSSolver, 
    NLISolver, 
    ASESolver, 
    OSNRCalculator, 
    OSNRCalculatorV2
)
from CFM.core.post_process import GSNRPlotter
from CFM.utils.build_alpha_db import build_alpha_for_band, load_alpha_reference, load_alpha_db_lcs

# Define and create results directory
results_dir = base_dir.parent / "results" / "single_link"
results_dir.mkdir(parents=True, exist_ok=True)



### 1. Multi-Band Spectrum & Attenuation Synthesis

Configure an ultra-wideband grid across the **E, S, C, and L bands**[cite: 2]:
* **Transceiver Settings:** 52 GBaud symbol rate over a 75 GHz channel grid[cite: 1].
* **Spectral Grid:** Descending frequency allocation across all bands.
* **Fiber Attenuation:** Dynamically interpolated per-channel via `build_alpha_for_band`[cite: 8].

In [30]:
channel_spacing_thz = 0.075  # 75 GHz spacing
baud_rate = 52e9             # 52 GBaud

# Define Optical Parameters per band
opt_l = OpticalParameters(Rs_mat=baud_rate)
opt_c = OpticalParameters(Rs_mat=baud_rate)
opt_s = OpticalParameters(Rs_mat=baud_rate)
opt_e = OpticalParameters(Rs_mat=baud_rate)

# Construct optical transmission bands
band_l = Band(
    name="l",
    start_freq=(184.539 - channel_spacing_thz / 2),
    end_freq=(190.464 + channel_spacing_thz / 2),
    opt_params=opt_l,
    channel_spacing=channel_spacing_thz
)

band_c = Band(
    name="c",
    start_freq=(190.914 - channel_spacing_thz / 2),
    end_freq=(196.839 + channel_spacing_thz / 2),
    opt_params=opt_c,
    channel_spacing=channel_spacing_thz
)

band_s = Band(
    name="s",
    start_freq=(197.289 - channel_spacing_thz / 2),
    end_freq=(203.514 + channel_spacing_thz / 2),
    opt_params=opt_s,
    channel_spacing=channel_spacing_thz
)

band_e = Band(
    name="e",
    start_freq=(204.000 - channel_spacing_thz / 2),
    end_freq=(209.000 + channel_spacing_thz / 2),
    opt_params=opt_e,
    channel_spacing=channel_spacing_thz
)

# Ordered bands list (descending frequency order)
bands = [band_e, band_s, band_c, band_l]

# Build unified carrier grid (Hz) across all active bands
grid_center = np.concatenate([b.calc_spectrum() + channel_spacing_thz * 0.5 for b in bands])
grid_center = grid_center[::-1] * 1e12

print(f"Total Transmission Channels Configured: {len(grid_center)}")
for b in bands:
    print(f" - Band {b.name.upper()}: {b.num_channels} channels | NF: {b.noise_figure:.1f} (linear)")
alpha_ref, freq_ref = load_alpha_reference()

# Generate channel-by-channel attenuation across all bands
alpha_list = [build_alpha_for_band(alpha_ref, freq_ref, b) for b in bands]
alpha_dB_custom = np.concatenate(alpha_list, axis=1)

print(f"Interpolated attenuation vector shape: {alpha_dB_custom.shape}")
print(f"Mean attenuation: {np.mean(alpha_dB_custom):.3f} dB/km | Min: {np.min(alpha_dB_custom):.3f} dB/km | Max: {np.max(alpha_dB_custom):.3f} dB/km")



Total Transmission Channels Configured: 313
 - Band E: 68 channels | NF: 5.0 (linear)
 - Band S: 85 channels | NF: 6.0 (linear)
 - Band C: 80 channels | NF: 4.5 (linear)
 - Band L: 80 channels | NF: 5.0 (linear)
Interpolated attenuation vector shape: (1, 313)
Mean attenuation: 0.202 dB/km | Min: 0.188 dB/km | Max: 0.232 dB/km


### 2. Single-Span Analysis: Ideal Power Allocation

Model a single 80 km SSMF span under flat launch conditions:
* **Span Length:** 80 km SSMF[cite: 4].
* **Launch Power:** Flat 0.0 dBm per channel (`ideal_power_management=True`)[cite: 6].
* **Solvers:** Forward Launch Power (FLP) ISRS[cite: 6], closed-form NLI[cite: 6], and accumulated ASE[cite: 6].

In [32]:
link_params = LinkParameters()

single_span_link = Link(
    name="single_span_80km",
    length=[80.0],
    num_span=1,
    num_amp=1,
    link_params=link_params
)

# Case A: Ideal power management (flat 0.0 dBm per channel launch power)
ideal_param_builder = ParameterBuilder(
    link=single_span_link,
    bands=bands,
    P_in=2.0,
    grid_center=grid_center,
    alpha_dB_LCS=alpha_dB_custom,
    P_in_is_tx_power=False
)

# Solve ISRS, NLI, and ASE for the ideal case
isrs_ideal = ISRSSolver(ideal_param_builder, model="FLP")
p_in, p_out, a0, a1, sig = isrs_ideal.solve()

nli_ideal = NLISolver(ideal_param_builder, a0, a1, sig).solve()
ase_ideal = ASESolver(ideal_param_builder).solve()

osnr_ideal = OSNRCalculator(ideal_param_builder, ase_ideal, nli_ideal)
on_ideal, oa_ideal, ot_ideal = osnr_ideal.compute()

print(f"Ideal Launch - Mean GSNR: {np.mean(ot_ideal[0]):.2f} dB (Total), {np.mean(oa_ideal[0]):.2f} dB (Linear), {np.mean(on_ideal[0]):.2f} dB (Nonlinear)")

# Plot Single-Span GSNR Profiles (0-based indexing: N_s_max=0)
curves = [
    {"name": "GSNR Total", "values": ot_ideal, "color": "blue"},
    {"name": "GSNR Linear (ASE)", "values": oa_ideal, "color": "green"},
    {"name": "GSNR Non-linear (NLI)", "values": on_ideal, "color": "red"},
]

plotter = GSNRPlotter(ideal_param_builder, curves, bands=bands)
fig = plotter.plot(N_s_max=0, matlab_indexing=False)
fig.update_layout(title="Single-Span GSNR Spectral Profile (Ideal Launch 0 dBm)")
fig.show()


Ideal Launch - Mean GSNR: 27.49 dB (Total), 29.25 dB (Linear), 33.75 dB (Nonlinear)


### 3. Multi-Span Cascade: Physical EDFAs & Tilt Modeling

Evaluate a 3-span heterogeneous cascade with physical amplifier imperfections:
* **Link Profile:** 80 km + 70 km + 60 km spans[cite: 1, 4].
* **Amplification:** Post-transmitter booster (+14 dB) followed by inline EDFAs[cite: 1, 3].
* **Gain Shaping:** Per-span tilt targets to compensate ISRS spectral ripple[cite: 3, 6].

In [36]:

multi_span_link = Link(
    name="multi_span_cascade",
    length=[80.0, 70.0, 60.0],
    num_span=3,
    num_amp=3,
    link_params=link_params
)

booster_edfa = EdfaConfig(gain_target=14.0, is_booster=True)

# Define physical inline amplifiers with slight tilt compensation
inline_edfas = [
    EdfaConfig(gain_target=16.0, tilt_target=-0.5, out_voa=0.5),
    EdfaConfig(gain_target=14.5, tilt_target=-0.3, out_voa=0.0),
    EdfaConfig(gain_target=13.0, tilt_target=0.0, out_voa=0.0),
]

# Physical launch configuration
phys_param_builder = ParameterBuilder(
    link=multi_span_link,
    bands=bands,
    P_in=-12.0,                  # Transmitter power prior to booster
    grid_center=grid_center,
    Edfa=inline_edfas,
    booster=booster_edfa,
    P_in_is_tx_power=True,      # Apply booster gain to P_in
    alpha_dB_LCS=alpha_dB_custom
)

# Execute Solvers across the cascade
isrs_phys = ISRSSolver(phys_param_builder, model="FLP")
p_in_cascade, p_out_cascade, a0_c, a1_c, sig_c = isrs_phys.solve()

nli_cascade = NLISolver(phys_param_builder, a0_c, a1_c, sig_c).solve()
ase_cascade = ASESolver(phys_param_builder).solve()

# Extended OSNR calculation (V2)
osnr_v2 = OSNRCalculatorV2(phys_param_builder, ase_cascade, nli_cascade)
res_cascade = osnr_v2.compute()

# Visualizing GSNR at the end of the final span (Span 3 -> Index 2)
curves_cascade = [
    {"name": "Total GSNR", "values": res_cascade["OSNR_NLI_ASE_dB"], "color": "#1f77b4"},
    {"name": "ASE OSNR", "values": res_cascade["OSNR_ASE_dB"], "color": "#2ca02c"},
    {"name": "NLI OSNR", "values": res_cascade["OSNR_NLI_dB"], "color": "#d62728"},
]

plotter_cascade = GSNRPlotter(phys_param_builder, curves_cascade, bands=bands)
fig_cascade = plotter_cascade.plot(N_s_max=2, matlab_indexing=False)
fig_cascade.update_layout(title="Multi-Span Cascade GSNR Profile (End of Span 3)")
fig_cascade.show()

